In [1]:
import scipy as sp
import numpy as np
import pandas as pd
import pickle
import os

from tqdm.notebook import tqdm

# pd.set_option('display.max_rows', 50)

In [2]:
def mean_confidence_interval(x, confidence=0.95):
    m, se = np.mean(x), sp.stats.sem(x)
    h = se * sp.stats.t.ppf((1 + confidence) / 2., x.shape[0] - 1)
    return np.round(m - h, 2), np.round(m, 2), np.round(m + h, 2)

In [3]:
folder_path = "./new_sweep_results/"
files = [folder_path + entry.name for entry in os.scandir(folder_path) if (entry.is_file() and "loc2" in entry.name)]
print(files)

['./new_sweep_results/sweep_loc2_30_0.001_128.pkl', './new_sweep_results/sweep_loc2_30_0.001_32.pkl', './new_sweep_results/sweep_loc2_2_0.0001_32.pkl', './new_sweep_results/sweep_loc2_2_0.0001_64.pkl', './new_sweep_results/sweep_loc2_20_0.0001_128.pkl', './new_sweep_results/sweep_loc2_10_0.001_128.pkl', './new_sweep_results/sweep_loc2_10_0.001_32.pkl', './new_sweep_results/sweep_loc2_10_0.0001_64.pkl', './new_sweep_results/sweep_loc2_30_0.0001_32.pkl', './new_sweep_results/sweep_loc2_30_0.001_64.pkl', './new_sweep_results/sweep_loc2_20_0.001_64.pkl', './new_sweep_results/sweep_loc2_2_0.001_64.pkl', './new_sweep_results/sweep_loc2_10_0.001_64.pkl', './new_sweep_results/sweep_loc2_5_0.001_64.pkl', './new_sweep_results/sweep_loc2_5_0.0001_64.pkl', './new_sweep_results/sweep_loc2_20_0.001_128.pkl', './new_sweep_results/sweep_loc2_2_0.001_32.pkl', './new_sweep_results/sweep_loc2_10_0.0001_128.pkl', './new_sweep_results/sweep_loc2_5_0.001_128.pkl', './new_sweep_results/sweep_loc2_5_0.0001_32

In [4]:
final = pd.DataFrame()
for file in tqdm(files):
    with open(file, "rb") as f:
        results = pickle.load(f)

    keys = [x for x in results.keys() if "mask" not in x and "true" not in x]
    processed_results = {key: {e: [] for e in ["value", "time"]} for key in keys}
    
    for key in keys:
        tmp_mean = np.mean((results[key]["value"] - results["- - true"]["value"][:, :1]) ** 2, axis=0)
        processed_results[key]["value"] = sorted(list(tmp_mean))
        processed_results[key]["time"] = sorted(results[key]["time"])

    tmp = pd.DataFrame(processed_results).transpose()
    tmp.reset_index(inplace=True)
    
    tmp["latent_dim"] = tmp["index"].apply(lambda x: x.split()[0]).astype(int)
    tmp["time_feature_dim"] = tmp["index"].apply(lambda x: x.split()[1]).astype(int)
    tmp["score_type"] = tmp["index"].apply(lambda x: x.split()[2])
    
    tmp["value"] = tmp.value.apply(lambda x: np.round(np.array(x), 2))
    tmp["time"] = tmp.time.apply(lambda x: np.round(np.array(x), 2))
    
    special = file.split("/")[-1][:-4].split("_")[2:]
    tmp["num_dims"] = int(special[0])
    tmp["lr"] = float(special[1])
    tmp["cond_latent_dim"] = int(special[2])
    
    tmp.drop("index", inplace=True, axis=1)
    tmp = tmp[["num_dims", "lr", "time_feature_dim", "latent_dim", "cond_latent_dim", "score_type", "value", "time"]]

    final = pd.concat([final, tmp])
final.sort_values(["num_dims", "lr", "time_feature_dim", "latent_dim", "cond_latent_dim", "score_type"], ascending=True, inplace=True)
final

  0%|          | 0/30 [00:00<?, ?it/s]

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time
0,2,0.0001,1,32,32,naive,"[0.18, 0.22, 0.27]","[139.21, 141.01, 144.88]"
1,2,0.0001,1,32,32,rl,"[0.05, 0.07, 0.08]","[45.16, 48.41, 50.76]"
0,2,0.0001,1,32,64,naive,"[0.08, 0.14, 0.43]","[141.92, 142.3, 143.96]"
1,2,0.0001,1,32,64,rl,"[0.02, 0.04, 0.24]","[51.73, 54.2, 55.64]"
0,2,0.0001,1,32,128,naive,"[0.18, 0.21, 0.29]","[149.91, 150.83, 151.47]"
...,...,...,...,...,...,...,...,...
39,30,0.0010,128,512,32,rl,"[222.65, 315.81, 396.43]","[111.01, 113.62, 118.21]"
38,30,0.0010,128,512,64,naive,"[83.18, 179.19, 739.07]","[94.59, 99.34, 111.07]"
39,30,0.0010,128,512,64,rl,"[140.37, 351.5, 445.27]","[44.56, 48.49, 52.83]"
38,30,0.0010,128,512,128,naive,"[15.94, 324.4, 656.19]","[115.08, 122.49, 126.47]"


In [5]:
final["mean_value"] = final.value.apply(np.mean)
final["var_value"] = final.value.apply(np.std)
final["mean_time"] = final.time.apply(np.mean)

/home/icb/egor.antipov/miniconda3/envs/scaling-transformers/lib/python3.10/site-packages/numpy/_core/_methods.py:185: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


In [6]:
final[final.num_dims == 2].sort_values("mean_value").head(10)

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time,mean_value,var_value,mean_time
3,2,0.0001,32,32,64,rl,"[0.03, 0.04, 0.05]","[45.59, 49.06, 53.23]",0.040000,0.008165,49.293333
25,2,0.0001,1,256,128,rl,"[0.01, 0.06, 0.07]","[60.51, 64.4, 66.65]",0.046667,0.026247,63.853333
29,2,0.0001,64,256,64,rl,"[0.04, 0.05, 0.05]","[56.49, 57.46, 63.78]",0.046667,0.004714,59.243333
25,2,0.0001,1,256,64,rl,"[0.04, 0.05, 0.06]","[54.55, 56.17, 59.55]",0.050000,0.008165,56.756667
1,2,0.0001,1,32,128,rl,"[0.03, 0.05, 0.07]","[48.35, 56.28, 56.46]",0.050000,0.016330,53.696667
19,2,0.0001,32,128,128,rl,"[0.03, 0.04, 0.09]","[51.88, 55.87, 65.99]",0.053333,0.026247,57.913333
35,2,0.0001,32,512,64,rl,"[0.03, 0.06, 0.07]","[57.9, 65.11, 70.73]",0.053333,0.016997,64.580000
9,2,0.0001,1,64,32,rl,"[0.02, 0.07, 0.07]","[50.26, 52.24, 53.28]",0.053333,0.023570,51.926667
29,2,0.0001,64,256,128,rl,"[0.04, 0.04, 0.09]","[55.03, 56.7, 59.76]",0.056667,0.023570,57.163333
23,2,0.0001,128,128,128,rl,"[0.05, 0.06, 0.06]","[52.4, 58.77, 61.23]",0.056667,0.004714,57.466667


In [7]:
final[final.num_dims == 5].sort_values("mean_value").head(10)

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time,mean_value,var_value,mean_time
35,5,0.0001,32,512,128,rl,"[0.06, 0.08, 0.2]","[72.9, 75.18, 80.85]",0.113333,0.061824,76.310000
25,5,0.0001,1,256,64,rl,"[0.1, 0.11, 0.22]","[22.67, 24.71, 25.57]",0.143333,0.054365,24.316667
1,5,0.0001,1,32,64,rl,"[0.12, 0.17, 0.22]","[18.02, 20.99, 21.31]",0.170000,0.040825,20.106667
17,5,0.0001,1,128,64,rl,"[0.08, 0.13, 0.32]","[20.82, 21.54, 26.58]",0.176667,0.103387,22.980000
9,5,0.0001,1,64,128,rl,"[0.1, 0.11, 0.34]","[20.95, 21.24, 22.74]",0.183333,0.110855,21.643333
1,5,0.0001,1,32,32,rl,"[0.12, 0.17, 0.28]","[17.32, 19.43, 20.23]",0.190000,0.066833,18.993333
25,5,0.0001,1,256,128,rl,"[0.16, 0.2, 0.22]","[24.29, 26.02, 26.4]",0.193333,0.024944,25.570000
35,5,0.0001,32,512,64,rl,"[0.14, 0.19, 0.25]","[25.38, 26.36, 27.89]",0.193333,0.044969,26.543333
5,5,0.0001,64,32,64,rl,"[0.14, 0.16, 0.28]","[19.38, 20.5, 23.98]",0.193333,0.061824,21.286667
29,5,0.0001,64,256,128,rl,"[0.1, 0.23, 0.25]","[22.95, 24.63, 25.76]",0.193333,0.066500,24.446667


In [8]:
final[final.num_dims == 10].sort_values("mean_value").head(10)

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time,mean_value,var_value,mean_time
3,10,0.0001,32,32,32,rl,"[0.52, 0.52, 0.64]","[16.22, 16.68, 17.84]",0.560000,0.056569,16.913333
13,10,0.0001,64,64,64,rl,"[0.41, 0.6, 0.73]","[78.49, 96.21, 104.6]",0.580000,0.131403,93.100000
11,10,0.0001,32,64,32,rl,"[0.52, 0.55, 0.86]","[16.51, 18.56, 18.85]",0.643333,0.153695,17.973333
3,10,0.0001,32,32,64,rl,"[0.49, 0.68, 0.86]","[79.9, 90.55, 90.9]",0.676667,0.151070,87.116667
29,10,0.0001,64,256,64,rl,"[0.45, 0.67, 0.99]","[89.71, 95.9, 109.62]",0.703333,0.221711,98.410000
35,10,0.0001,32,512,128,rl,"[0.61, 0.71, 0.87]","[100.36, 119.59, 127.3]",0.730000,0.107083,115.750000
37,10,0.0001,64,512,32,rl,"[0.47, 0.65, 1.1]","[17.81, 18.24, 18.36]",0.740000,0.264953,18.136667
9,10,0.0001,1,64,128,rl,"[0.38, 0.59, 1.28]","[91.13, 96.52, 125.26]",0.750000,0.384448,104.303333
29,10,0.0001,64,256,32,rl,"[0.64, 0.77, 0.88]","[15.56, 17.79, 19.47]",0.763333,0.098093,17.606667
1,10,0.0001,1,32,64,rl,"[0.5, 0.63, 1.21]","[82.77, 83.04, 102.63]",0.780000,0.308653,89.480000


In [9]:
final[final.num_dims == 20].sort_values("mean_value").head(10)

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time,mean_value,var_value,mean_time
9,20,0.0001,1,64,128,rl,"[4.3, 7.43, 9.22]","[47.24, 49.44, 51.14]",6.983333,2.033262,49.273333
33,20,0.0001,1,512,64,rl,"[3.42, 8.45, 9.54]","[52.07, 62.96, 67.34]",7.136667,2.665487,60.790000
17,20,0.0001,1,128,32,rl,"[4.5, 4.68, 12.33]","[71.02, 75.12, 97.79]",7.170000,3.649411,81.310000
9,20,0.0001,1,64,32,rl,"[4.66, 5.35, 12.55]","[75.99, 77.38, 78.41]",7.520000,3.567885,77.260000
13,20,0.0001,64,64,32,rl,"[3.65, 7.87, 11.37]","[70.55, 77.19, 81.83]",7.630000,3.156242,76.523333
9,20,0.0001,1,64,64,rl,"[3.53, 5.44, 13.94]","[44.24, 45.36, 46.43]",7.636667,4.524823,45.343333
17,20,0.0001,1,128,64,rl,"[2.77, 9.48, 11.09]","[42.51, 45.65, 46.64]",7.780000,3.603064,44.933333
25,20,0.0001,1,256,32,rl,"[5.31, 8.24, 10.0]","[84.37, 90.45, 92.01]",7.850000,1.934442,88.943333
3,20,0.0001,32,32,64,rl,"[2.96, 10.01, 10.77]","[39.62, 41.81, 45.88]",7.913333,3.516251,42.436667
17,20,0.0001,1,128,128,rl,"[6.45, 6.49, 11.23]","[47.01, 49.26, 50.74]",8.056667,2.243945,49.003333


In [10]:
final[final.num_dims == 30].sort_values("mean_value").head(10)

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time,mean_value,var_value,mean_time
39,30,0.0001,128,512,128,rl,"[11.52, 22.29, 32.6]","[82.06, 93.05, 93.88]",22.136667,8.606557,89.663333
1,30,0.0001,1,32,128,rl,"[19.95, 27.99, 28.0]","[82.68, 83.93, 84.97]",25.313333,3.792452,83.860000
9,30,0.0001,1,64,64,rl,"[23.09, 26.58, 32.78]","[30.17, 32.45, 34.22]",27.483333,4.007163,32.280000
23,30,0.0001,128,128,128,rl,"[25.22, 28.92, 35.8]","[85.46, 89.76, 90.39]",29.980000,4.383819,88.536667
16,30,0.0010,1,128,32,naive,"[17.76, 18.93, 53.76]","[186.22, 188.81, 191.48]",30.150000,16.701623,188.836667
22,30,0.0001,128,128,128,naive,"[29.91, 33.48, 35.28]","[195.05, 211.46, 213.58]",32.890000,2.231636,206.696667
25,30,0.0001,1,256,32,rl,"[25.84, 33.15, 39.73]","[33.58, 35.19, 38.72]",32.906667,5.673179,35.830000
31,30,0.0001,128,256,128,rl,"[7.32, 37.89, 53.68]","[79.2, 87.52, 90.71]",32.963333,19.244331,85.810000
9,30,0.0001,1,64,32,rl,"[13.51, 38.29, 47.56]","[27.37, 30.32, 31.11]",33.120000,14.373524,29.600000
17,30,0.0001,1,128,32,rl,"[34.29, 34.42, 34.8]","[27.74, 29.72, 32.17]",34.503333,0.216384,29.876667


In [11]:
final.to_csv("sweep_results.csv", index=None)

In [19]:
tmp = final[final.num_dims == 2].sort_values("mean_value").head(20)[["time_feature_dim", "latent_dim", "cond_latent_dim"]].copy()
set_2 = set(map(tuple, list(tmp.values)))

tmp = final[final.num_dims == 5].sort_values("mean_value").head(20)[["time_feature_dim", "latent_dim", "cond_latent_dim"]].copy()
set_5 = set(map(tuple, list(tmp.values)))

tmp = final[final.num_dims == 10].sort_values("mean_value").head(20)[["time_feature_dim", "latent_dim", "cond_latent_dim"]].copy()
set_10 = set(map(tuple, list(tmp.values)))

tmp = final[final.num_dims == 20].sort_values("mean_value").head(20)[["time_feature_dim", "latent_dim", "cond_latent_dim"]].copy()
set_20 = set(map(tuple, list(tmp.values)))

tmp = final[final.num_dims == 30].sort_values("mean_value").head(20)[["time_feature_dim", "latent_dim", "cond_latent_dim"]].copy()
set_30 = set(map(tuple, list(tmp.values)))

In [20]:
set_2.intersection(set_5).intersection(set_10).intersection(set_20).intersection(set_30)

{(np.int64(1), np.int64(64), np.int64(32))}

In [34]:
final[final.num_dims == 20].sort_values("mean_value").head(30)

,num_dims,lr,time_feature_dim,latent_dim,cond_latent_dim,score_type,value,time,mean_value,var_value
11,20,0.0001,32,64,32,rl,"[2.16, 3.64, 5.15]","[12.81, 13.97, 14.45]",3.650000,1.220683
11,20,0.0001,32,64,64,rl,"[2.54, 3.77, 6.12]","[13.87, 14.83, 14.99]",4.143333,1.485179
27,20,0.0001,32,256,32,rl,"[3.76, 4.01, 5.89]","[14.64, 14.73, 15.15]",4.553333,0.950661
17,20,0.0001,1,128,32,rl,"[2.56, 5.44, 6.91]","[13.15, 13.34, 15.19]",4.970000,1.806710
17,20,0.0001,1,128,64,rl,"[2.85, 4.45, 7.64]","[15.09, 15.34, 15.58]",4.980000,1.991097
9,20,0.0001,1,64,64,rl,"[3.41, 5.85, 7.66]","[14.15, 14.36, 14.53]",5.640000,1.741398
27,20,0.0001,32,256,64,rl,"[3.93, 6.27, 8.23]","[16.94, 17.93, 18.24]",6.143333,1.757751
17,20,0.0001,1,128,128,rl,"[6.08, 6.34, 6.74]","[13.94, 15.92, 16.66]",6.386667,0.271457
9,20,0.0001,1,64,128,rl,"[4.51, 4.88, 10.22]","[14.8, 15.16, 16.18]",6.536667,2.608887
37,20,0.0001,64,512,128,rl,"[4.27, 7.95, 8.16]","[16.74, 16.87, 20.69]",6.793333,1.786325
